# Lab 1 · EDA and simple baselines
Pairs · 24 September 2026, 12:30 Santiago · 6% NP.
Read the brief and rubric first. Supplied code supports execution; replace the written prompts and add your own analysis cells. MODE="synthetic" is fictional practice only. Real submission uses snapshot.

In [69]:
from pathlib import Path
import sys, platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
for p in (Path.cwd(), *Path.cwd().parents):
    if (p/'labs/lab_01/lab01_support_v1.py').exists():
        ROOT=p; break
else:
    raise FileNotFoundError('Extract the full package and run inside it.')
sys.path.insert(0,str(ROOT/'labs/lab_01'))
from lab01_support_v1 import load_sources,merge_sources,majority_from_train,evaluate
MODE="snapshot"
print('DATA MODE:', MODE, '| Python:',platform.python_version(),'| pandas:',pd.__version__)

DATA MODE: snapshot | Python: 3.10.3 | pandas: 2.2.3


## 1. Target and audit
State the prediction moment, unit, population and target. Distinguish final classification from completing a race. Explain the source limitations.

### Prediction moment

We predict **after qualifying has finished and before the Grand Prix starts**. The only predictor is `qualifying_position` from the **qualifying table**. We do not use `grid` from the results table: it is published together with the race result, it already includes post-qualifying grid penalties and pit-lane starts (`grid = 0`), and the brief fixes qualifying position as the pre-race information. Position, points, status and laps are post-race information: we use them only to audit and explore the training data, never as predictors.

### Unit

One observation is **one driver in one Grand Prix**, identified by the key `(season, round, driver_id)`.

### Population

The population is **every driver/race record in the supplied race-results snapshots**. For training that means all drivers in 2019–2021 (1,200 rows), not a single driver. We keep every row: retirements, non-finishers, withdrawn and disqualified entries, and entries with a missing qualifying record. This is a *retrospective* population built from who appears in the results table. It is not the entry list that was known before each race.

### Target

`target_top10 = 1` when the source's numeric final race position is 1–10, and `0` otherwise. This is **final classification**, which is not the same as:
- **completing the race**: a driver can retire and still be classified by the laps they completed, and a driver who finishes 11th or lower is a 0;
- **scoring points**: points also depend on scoring rules such as the fastest-lap bonus, half points and later penalties;
- winning, or a constructor result.

### Data limitations

The snapshot reflects results as they are recorded *today*, so it can include corrections made after a race (for example a post-race time penalty or a disqualification that moved a driver out of the top ten). It is therefore not an exact reconstruction of what was known at the historical prediction moment. The population is also retrospective: drivers who were entered but never appear in the results table are invisible to us. Finally, a few results rows have no qualifying record, so the qualifying rule has to fall back to the training majority for them.

In [70]:
q_train,r_train=load_sources('train',ROOT,MODE)
train=merge_sources(q_train,r_train)
display(pd.DataFrame({'qualifying':q_train.groupby('season').size(),'results':r_train.groupby('season').size()}))
display(train.groupby('season').agg(rows=('driver_id','size'),missing_qualifying=('qualifying_position',lambda x:x.isna().sum())))
assert len(train)==len(r_train)
display(train['qualifying_match'].value_counts())

,qualifying,results
season,,
2019,418,420
2020,340,340
2021,439,440


,rows,missing_qualifying
season,,
2019,420,2
2020,340,0
2021,440,1


qualifying_match
both          1197
left_only        3
right_only       0
Name: count, dtype: int64

### Data integrity checks

We audit the **full training population** (all drivers, 2019–2021) and cover: keys, duplicates, unmatched records in both directions, missingness, coverage by season, final classification vs completing the race, and qualifying position vs grid.

In [74]:
# Audit of the full training population (all drivers, 2019-2021).
KEYS = ['season', 'round', 'driver_id']

# 1. Keys: nulls and duplicates in each source and in the merged table
key_check = pd.DataFrame({
    name: {'rows': len(df),
           'null_keys': int(df[KEYS].isna().any(axis=1).sum()),
           'duplicate_keys': int(df.duplicated(KEYS).sum())}
    for name, df in [('qualifying', q_train), ('results', r_train), ('merged', train)]
}).T
print('================================================================================')
print('Key audit: rows, null keys, duplicate keys')
display(key_check)


# 2. Unmatched records in both directions.
# A left join from results only shows results without qualifying (left_only). Qualifying rows with no
# race result can never appear in it, so we check them with an anti-join from the qualifying side.
q_side = q_train[KEYS].merge(r_train[KEYS], on=KEYS, how='left', indicator=True)
print('================================================================================')
print('Unmatched records:')
print('Results rows without a qualifying record (left_only):', int(train['qualifying_match'].eq('left_only').sum()))
print('Qualifying rows without a race result (qualifying-only):', int(q_side['_merge'].eq('left_only').sum()))

# 3. Missingness per column, and the rows that will need the missing-qualifying fallback
missing = train.isna().sum()
print('================================================================================')
print('Missing values per column:')
display(missing[missing > 0].rename('missing_rows').to_frame())
print('=================================================================================')
print('Rows with missing qualifying position (will need fallback):', int(train['qualifying_position'].isna().sum()))
display(train.loc[train['qualifying_position'].isna(),
                  ['season', 'round', 'race_name', 'driver_id', 'constructor_id', 'grid', 'position', 'status']])

# 4. Coverage by season
coverage = train.groupby('season').agg(
    races=('round', 'nunique'),
    drivers=('driver_id', 'nunique'),
    rows=('driver_id', 'size'),
    missing_qualifying=('qualifying_position', lambda x: int(x.isna().sum())),
    target_rate=('target_top10', 'mean'),
)
coverage['rows_per_race'] = coverage['rows'] / coverage['races']
print('=================================================================================')
print('Coverage by season:')
display(coverage.round(3))

# 5. Final classification vs completing the race (status is post-race: audit only, not a predictor)
completed = train['status'].eq('Finished') | train['status'].str.match(r'^\+\d+ Laps?$')
print('=================================================================================')
print('Rows with status indicating the driver completed the race:', int(completed.sum()), 'of', len(completed))
display(pd.crosstab(train['target_top10'], completed.map({True: 'completed race', False: 'did not complete'}),
                    margins=True))
print('=================================================================================')
print('Rows with non-numeric position_text (DNF, DNS, DSQ, etc.):', int(train['position_text'].str.isdigit().eq(False).sum()), 'of', len(train))
display(train['position_text'].astype(str).value_counts()
        .loc[lambda s: ~s.index.str.isdigit()].rename('rows').to_frame())
print('=================================================================================')
print('Rows with points > 0:', int(train['points'].gt(0).sum()), 'of', len(train))
display(pd.crosstab(train['target_top10'], train['points'].gt(0).map({True: 'points > 0', False: 'no points'})))


# 6. Qualifying position vs results grid (why grid is not our predictor)
has_q = train['qualifying_position'].notna()
print('=================================================================================')
print('Rows where grid differs from qualifying position:',
      int((train.loc[has_q, 'grid'] != train.loc[has_q, 'qualifying_position']).sum()), 'of', int(has_q.sum()))
print('Rows with grid = 0 (pit-lane start):', int(train['grid'].eq(0).sum()))

Key audit: rows, null keys, duplicate keys


,rows,null_keys,duplicate_keys
qualifying,1197,0,0
results,1200,0,0
merged,1200,0,0


Unmatched records:
Results rows without a qualifying record (left_only): 3
Qualifying rows without a race result (qualifying-only): 0
Missing values per column:


,missing_rows
qualifying_position,3


Rows with missing qualifying position (will need fallback): 3


...

Rows with non-numeric position_text (DNF, DNS, DSQ, etc.): 155 of 1200


,rows
position_text,
R,146
W,6
D,3


Rows with points > 0: 600 of 1200


points,no points,points > 0
target_top10,,
0,600,0
1,0,600


Rows where grid differs from qualifying position: 381 of 1197
Rows with grid = 0 (pit-lane start): 29


### Audit findings

- **Rows and keys:** qualifying has 418 / 340 / 439 rows and results 420 / 340 / 440 rows for 2019 / 2020 / 2021. No source has null or duplicate `(season, round, driver_id)` keys, and the merged table keeps exactly the 1,200 results rows (`assert len(train)==len(r_train)` passes).
- **Unmatched records:** 1,197 rows match (`both`). **3 results rows have no qualifying record** (`left_only`): two in the 2019 Chinese GP and one in the 2021 Monaco GP. **No qualifying row lacks a race result**, so no qualifying-only record exists in train.
- **Missingness:** `qualifying_position` is missing only for those 3 rows (2 in 2019, 1 in 2021). They stay in the population and will use the training-majority fallback in the qualifying baseline. One of them (2019 Chinese GP, pit-lane start, final position 10) is a target-1 row, so the fallback of 0 would produce a false negative on it.
- **Coverage:** 21 / 17 / 22 races, with exactly 20 rows per race in every season. 2020 is shorter (COVID calendar) and has 23 different drivers because of substitutes. The training target rate is **exactly 0.500** (600 of 1,200) because each race has 10 classified top-ten places out of 20 rows. The majority is therefore an exact tie, which the fixed rule resolves to **0**.
- **Classification vs completing the race:** 176 rows did not complete the race (146 `R`, 6 `W`, 3 `D`, plus other non-finishing statuses). In 2019–2021 none of them was classified in the top ten, but they are still kept as target-0 rows. 424 drivers completed the race and still got a 0. In this period `points > 0` coincides with `target_top10` for all 1,200 rows. That is an observed fact for these seasons, not the definition we use.
- **Grid vs qualifying:** grid differs from qualifying position in 381 of the 1,197 rows that have qualifying (about 32%), and 29 rows have `grid = 0`. This confirms that `grid` carries post-qualifying information and must not replace the qualifying table.

**Decision:** keep all 1,200 rows, predict only from `qualifying_position`, and report the 3 fallback rows explicitly.

### Why the left join instead of the inner join?

The evaluated population is defined by the **results** table: every driver/race record that has a final classification. We therefore start from results and left-join qualifying on `(season, round, driver_id)`. An inner join would silently drop every results row without a qualifying record. In train that removes the 3 `left_only` rows (2019 Chinese GP ×2, 2021 Monaco GP ×1) and shrinks the population from 1,200 to 1,197. Those rows are not random: they are exactly the cases where the qualifying rule cannot be applied. Dropping them would hide the fallback, make the qualifying baseline look like it covers every row, and evaluate the two baselines on a smaller, filtered population instead of the one defined in the brief. The left join keeps them with a null `qualifying_position`, so the missing-qualifying policy stays visible and can be counted in calibration and test.

## 2. EDA question 1
Question: TO COMPLETE.

Use train only. Add a table/graph, cite observed values, interpret and state a decision or limitation. At least one of the three analyses must check a Friday trap.

In [ ]:
# Add your train-only analysis here.

## 3. EDA question 2
Question: TO COMPLETE.

Use train only. Add a table/graph, cite observed values, interpret and state a decision or limitation. At least one of the three analyses must check a Friday trap.

In [ ]:
# Add your train-only analysis here.

## 4. EDA question 3
Question: TO COMPLETE.

Use train only. Add a table/graph, cite observed values, interpret and state a decision or limitation. At least one of the three analyses must check a Friday trap.

In [ ]:
# Add your train-only analysis here.

## 5. Baselines and 2022 check
Explain the two fixed rules, tie handling and missing-qualifying fallback. Predict qualitatively how each might fail before running the check. TO COMPLETE.

In [ ]:
majority=majority_from_train(train)
print('Training majority:',majority,'| training target rate:',train.target_top10.mean())
q_cal,r_cal=load_sources('calibration',ROOT,MODE)
cal=merge_sources(q_cal,r_cal)
print('Calibration rows:',len(cal),'| missing qualifying:',cal.qualifying_position.isna().sum())
display(evaluate(cal,majority))

Interpret TN/FP/FN/TP, accuracy and balanced accuracy. Explain the positive/negative error costs in this task without inventing monetary values. Record any corrections after the check. TO COMPLETE.

## 6. Freeze before final test
Complete this record and commit the code before enabling RUN_TEST.

- Members and date: TO COMPLETE.
- Target, population, join and prediction moment: TO COMPLETE.
- Baselines, train majority, threshold and missing policy: TO COMPLETE.
- Three analysis decisions and changes after calibration: TO COMPLETE.
- Frozen code commit/version: TO COMPLETE.

The test switch is a learning aid. Accessing CSVs directly still counts as inspecting test. Do not change the method in response to test results. Document genuine later bug fixes transparently.

In [ ]:
RUN_TEST=False
if RUN_TEST:
    q_test,r_test=load_sources('test',ROOT,MODE,frozen=True)
    test=merge_sources(q_test,r_test)
    print('Test rows:',len(test),'| missing qualifying:',test.qualifying_position.isna().sum())
    display(evaluate(test,majority))
else:
    print('Test remains unopened by this notebook. Complete and commit the freeze record first.')

## 7. Final interpretation and handoff
Compare calibration and test, cite your observed results, explain one failure mode and what the evidence supports. Discuss the retrospective population and source timing limitation. TO COMPLETE.

Complete RUNBOOK and PROMPTS/contributions; restart/run all and save outputs. Submit the repository URL and final commit via Canvas. No separate report is required.